In [1]:
from typing import List, Optional, Tuple


class BTreeNode:
    """B树节点类"""
    def __init__(self, leaf: bool = True):
        self.leaf: bool = leaf          # 是否为叶子节点
        self.keys: List[int] = []       # 键值列表（从小到大排序）
        self.children: List[BTreeNode] = []  # 子节点列表（比keys多一个）

    def __repr__(self):
        return f"Node(keys={self.keys}, leaf={self.leaf})"


class BTree:
    """B树类（3阶，即每个节点最多有3个子节点，最多2个键）"""

    def __init__(self, order: int = 3):
        self.order: int = order          # 阶数
        self.max_keys: int = order - 1   # 最大键数 = 2
        self.min_keys: int = (order // 2) - 1 if order % 2 == 0 else (order // 2)  # 最小键数 = 1
        self.root: BTreeNode = BTreeNode(leaf=True)

    def insert(self, key: int, verbose: bool = True) -> None:
        """插入键值"""
        if verbose:
            print(f"\n{'='*60}")
            print(f"插入: {key}")
            print(f"{'='*60}")

        root = self.root
        # 如果根节点已满，需要分裂
        if len(root.keys) == self.max_keys:
            if verbose:
                print(f"根节点已满，需要分裂")
            new_root = BTreeNode(leaf=False)
            new_root.children.append(self.root)
            self._split_child(new_root, 0, verbose)
            self.root = new_root
            self._insert_non_full(self.root, key, verbose)
        else:
            self._insert_non_full(self.root, key, verbose)

        if verbose:
            self.print_tree()

    def _insert_non_full(self, node: BTreeNode, key: int, verbose: bool = True) -> None:
        """向未满的节点插入键值"""
        i = len(node.keys) - 1

        if node.leaf:
            # 叶子节点：直接插入
            node.keys.append(0)
            while i >= 0 and key < node.keys[i]:
                node.keys[i + 1] = node.keys[i]
                i -= 1
            node.keys[i + 1] = key
            if verbose:
                print(f"  插入到叶子节点: {node.keys}")
        else:
            # 内部节点：找到合适的子节点
            while i >= 0 and key < node.keys[i]:
                i -= 1
            i += 1

            if verbose:
                print(f"  进入第 {i} 个子节点")

            # 检查子节点是否已满
            if len(node.children[i].keys) == self.max_keys:
                if verbose:
                    print(f"  子节点 {node.children[i].keys} 已满，需要分裂")
                self._split_child(node, i, verbose)
                # 分裂后，判断key应该插入左子节点还是右子节点
                if key > node.keys[i]:
                    i += 1

            self._insert_non_full(node.children[i], key, verbose)

    def _split_child(self, parent: BTreeNode, index: int, verbose: bool = True) -> None:
        """
        分裂子节点
        parent: 父节点
        index: 要分裂的子节点在parent.children中的索引
        """
        order = self.order
        child = parent.children[index]

        # 创建新节点
        new_child = BTreeNode(leaf=child.leaf)

        # 将中间键提升到父节点
        mid_index = order // 2 - 1 if order % 2 == 0 else order // 2
        mid_key = child.keys[mid_index]

        if verbose:
            print(f"    分裂节点 {child.keys}，中间键 = {mid_key}")

        # 新节点获取原节点的右半部分键
        new_child.keys = child.keys[mid_index + 1:]
        # 原节点保留左半部分键
        child.keys = child.keys[:mid_index]

        # 如果不是叶子节点，还需要分裂子节点
        if not child.leaf:
            new_child.children = child.children[mid_index + 1:]
            child.children = child.children[:mid_index + 1]

        # 将中间键插入父节点
        parent.keys.insert(index, mid_key)
        parent.children.insert(index + 1, new_child)

        if verbose:
            print(f"    分裂后左子节点: {child.keys}")
            print(f"    分裂后右子节点: {new_child.keys}")
            print(f"    父节点变为: {parent.keys}")

    def inorder_traversal(self, node: BTreeNode = None) -> List[int]:
        """中序遍历"""
        if node is None:
            node = self.root

        result = []
        if node.leaf:
            result.extend(node.keys)
        else:
            for i in range(len(node.keys)):
                result.extend(self.inorder_traversal(node.children[i]))
                result.append(node.keys[i])
            result.extend(self.inorder_traversal(node.children[-1]))

        return result

    def print_tree(self, node: BTreeNode = None, level: int = 0, prefix: str = ""):
        """打印树形结构"""
        if node is None:
            node = self.root

        indent = "    " * level
        print(f"{indent}┌─ 节点 (keys={node.keys}, leaf={node.leaf})")

        if not node.leaf and node.children:
            for i, child in enumerate(node.children):
                if i < len(node.keys):
                    print(f"{indent}│  键 {node.keys[i]} 的左侧子树:")
                self.print_tree(child, level + 1, f"{indent}│  ")
                if i < len(node.keys):
                    print(f"{indent}├─ 键 {node.keys[i]} 的右侧子树:")
            # 打印最后一个子节点
            if node.children:
                self.print_tree(node.children[-1], level + 1, f"{indent}   ")

    def get_tree_structure(self, node: BTreeNode = None, level: int = 0) -> List[str]:
        """获取树形结构的文本表示"""
        if node is None:
            node = self.root

        lines = []
        indent = "  " * level

        # 显示节点信息
        keys_str = ", ".join(str(k) for k in node.keys)
        if node.leaf:
            node_type = "叶子"
        else:
            node_type = "内部"

        lines.append(f"{indent}[{node_type}节点] 键值: [{keys_str}] | 子节点数: {len(node.children)}")

        if not node.leaf and node.children:
            for i, child in enumerate(node.children):
                child_lines = self.get_tree_structure(child, level + 1)
                # 添加连接线
                if i < len(node.keys):
                    prefix = f"{indent}  键 {node.keys[i]} 下: "
                else:
                    prefix = f"{indent}  最后一个子节点: "
                child_lines[0] = prefix + child_lines[0].lstrip()
                lines.extend(child_lines)

        return lines

    def verify_properties(self) -> Tuple[bool, List[str]]:
        """
        验证B树的所有性质
        返回: (是否满足所有性质, 性质检查结果列表)
        """
        results = []
        all_passed = True

        # 性质1: 所有叶子节点在同一层
        leaf_depths = []

        def get_leaf_depths(node: BTreeNode, depth: int):
            if node.leaf:
                leaf_depths.append(depth)
            else:
                for child in node.children:
                    get_leaf_depths(child, depth + 1)

        get_leaf_depths(self.root, 0)
        leaves_same_level = len(set(leaf_depths)) == 1
        results.append(f"性质1 - 所有叶子节点在同一层: {'✓' if leaves_same_level else '✗'} (深度: {set(leaf_depths)})")
        all_passed = all_passed and leaves_same_level

        # 性质2: 每个节点中的键值按升序排列
        def check_keys_sorted(node: BTreeNode) -> bool:
            for i in range(1, len(node.keys)):
                if node.keys[i] <= node.keys[i-1]:
                    return False
            for child in node.children:
                if not check_keys_sorted(child):
                    return False
            return True

        keys_sorted = check_keys_sorted(self.root)
        results.append(f"性质2 - 节点内键值升序排列: {'✓' if keys_sorted else '✗'}")
        all_passed = all_passed and keys_sorted

        # 性质3: 根节点性质
        root_min_keys = 0  # 根节点至少0个键
        root_max_keys = self.max_keys
        root_keys_ok = len(self.root.keys) >= root_min_keys and len(self.root.keys) <= root_max_keys
        results.append(f"性质3 - 根节点键数范围 [0, {root_max_keys}]: {'✓' if root_keys_ok else '✗'} (实际: {len(self.root.keys)})")
        all_passed = all_passed and root_keys_ok

        # 性质4: 非根非叶子节点键数范围
        min_keys = self.min_keys
        max_keys = self.max_keys

        def check_non_root_keys(node: BTreeNode, is_root: bool) -> bool:
            if not is_root and not node.leaf:
                if len(node.keys) < min_keys or len(node.keys) > max_keys:
                    return False
            for child in node.children:
                if not check_non_root_keys(child, False):
                    return False
            return True

        non_root_keys_ok = check_non_root_keys(self.root, True)
        results.append(f"性质4 - 非根内部节点键数范围 [{min_keys}, {max_keys}]: {'✓' if non_root_keys_ok else '✗'}")
        all_passed = all_passed and non_root_keys_ok

        # 性质5: 叶子节点键数范围
        def check_leaf_keys(node: BTreeNode, is_root: bool) -> bool:
            if node.leaf:
                if not is_root:
                    if len(node.keys) < min_keys or len(node.keys) > max_keys:
                        return False
            for child in node.children:
                if not check_leaf_keys(child, False):
                    return False
            return True

        leaf_keys_ok = check_leaf_keys(self.root, True)
        results.append(f"性质5 - 非根叶子节点键数范围 [{min_keys}, {max_keys}]: {'✓' if leaf_keys_ok else '✗'}")
        all_passed = all_passed and leaf_keys_ok

        # 性质6: 子节点数 = 键数 + 1
        def check_child_count(node: BTreeNode) -> bool:
            if not node.leaf:
                if len(node.children) != len(node.keys) + 1:
                    return False
                for child in node.children:
                    if not check_child_count(child):
                        return False
            return True

        child_count_ok = check_child_count(self.root)
        results.append(f"性质6 - 子节点数 = 键数 + 1: {'✓' if child_count_ok else '✗'}")
        all_passed = all_passed and child_count_ok

        # 性质7: 键值有序性（左子树 < 键 < 右子树）
        def check_bst_property(node: BTreeNode) -> bool:
            for i, key in enumerate(node.keys):
                if i > 0 and key <= node.keys[i-1]:
                    return False
                if not node.leaf:
                    # 检查左子树的最大值 < 当前键
                    if node.children[i]:
                        max_left = get_max_key(node.children[i])
                        if max_left >= key:
                            return False
                    # 检查当前键 < 右子树的最小值
                    if node.children[i+1]:
                        min_right = get_min_key(node.children[i+1])
                        if min_right <= key:
                            return False
            for child in node.children:
                if not check_bst_property(child):
                    return False
            return True

        def get_max_key(node: BTreeNode) -> int:
            while not node.leaf:
                node = node.children[-1]
            return node.keys[-1]

        def get_min_key(node: BTreeNode) -> int:
            while not node.leaf:
                node = node.children[0]
            return node.keys[0]

        bst_ok = check_bst_property(self.root)
        results.append(f"性质7 - BST性质（左<根<右）: {'✓' if bst_ok else '✗'}")
        all_passed = all_passed and bst_ok

        return all_passed, results


# ==================== 主程序 ====================
def main():
    print("=" * 70)
    print("3阶 B-Tree 构建")
    print("插入序列: [10, 20, 5, 6, 12, 30, 25]")
    print("=" * 70)

    # 创建3阶B树
    btree = BTree(order=3)

    # 插入序列
    insert_sequence = [10, 20, 5, 6, 12, 30, 25]

    for key in insert_sequence:
        btree.insert(key, verbose=True)

    # 最终结果
    print("\n" + "=" * 70)
    print("最终 B-Tree")
    print("=" * 70)

    print("\n【树形结构】")
    structure_lines = btree.get_tree_structure()
    for line in structure_lines:
        print(line)

    print("\n【树的形态图】")
    print("""
                         [12]
                        /    \\
                      /        \\
                   [6]          [20, 30]
                  /   \\        /   |   \\
                [5]   [10]   [12] [25] [30]?

    ⚠️ 注意：上面的图是简化示意，实际B树结构如下：

    ┌─────────────────────────────────────────────────────────┐
    │                      根节点: [12]                        │
    │                      /         \\                        │
    │                 [6]              [20, 30]                │
    │                /   \\            /   |   \\               │
    │              [5]   [10]   [12的子?] [25] [30的子?]        │
    │                                                          │
    │  实际上，3阶B树每个节点最多2个键，最多3个子节点           │
    └─────────────────────────────────────────────────────────┘
    """)

    # 更精确的树形图
    print("\n【精确树形结构】")

    def print_fancy_tree(node, prefix="", is_last=True):
        if node is None:
            return

        # 打印当前节点
        connector = "└── " if is_last else "├── "
        keys_str = "[" + ", ".join(str(k) for k in node.keys) + "]"
        print(prefix + connector + keys_str)

        # 更新前缀
        child_prefix = prefix + ("    " if is_last else "│   ")

        # 打印子节点
        if not node.leaf and node.children:
            for i, child in enumerate(node.children):
                is_last_child = (i == len(node.children) - 1)
                print_fancy_tree(child, child_prefix, is_last_child)

    print_fancy_tree(btree.root)

    # 中序遍历验证
    print("\n【中序遍历】")
    inorder = btree.inorder_traversal()
    print(f"中序遍历结果: {inorder}")
    print(f"是否有序: {'✓' if inorder == sorted(inorder) else '✗'}")

    # 性质验证
    print("\n" + "=" * 70)
    print("B-Tree 性质验证")
    print("=" * 70)

    all_passed, results = btree.verify_properties()

    for result in results:
        print(f"  {result}")

    print("\n" + "-" * 70)
    print(f"最终结果: {'✓ 所有性质均满足，B-Tree构建正确！' if all_passed else '✗ 部分性质不满足，请检查！'}")
    print("-" * 70)

    # 详细构建过程说明
    print("\n" + "=" * 70)
    print("详细构建过程")
    print("=" * 70)
    print("""
┌─────────────────────────────────────────────────────────────────────┐
│ 步骤1: 插入 10                                                        │
│   -> 根节点: [10]                                                    │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤2: 插入 20                                                        │
│   -> 根节点: [10, 20] (未满)                                          │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤3: 插入 5                                                         │
│   -> 根节点: [5, 10, 20] (已满，需要分裂)                              │
│   -> 中间键 10 提升为新根                                             │
│   -> 分裂为: 根[10], 左子[5], 右子[20]                                │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤4: 插入 6                                                         │
│   -> 从根[10]开始，6 < 10，进入左子[5]                                 │
│   -> 左子[5]未满，插入后变为[5, 6]                                     │
│   -> 树结构: 根[10], 左子[5,6], 右子[20]                               │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤5: 插入 12                                                        │
│   -> 从根[10]开始，12 > 10，进入右子[20]                               │
│   -> 右子[20]未满，插入后变为[12, 20]                                   │
│   -> 树结构: 根[10], 左子[5,6], 右子[12,20]                            │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤6: 插入 30                                                        │
│   -> 从根[10]开始，30 > 10，进入右子[12,20]                            │
│   -> 右子[12,20]未满，插入后变为[12,20,30] (已满，需要分裂)             │
│   -> 右子分裂，中间键20提升到根                                        │
│   -> 根变为[10,20]，左子[5,6]，中子[12]，右子[30]                      │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤7: 插入 25                                                        │
│   -> 从根[10,20]开始                                                 │
│   -> 25 > 20，进入最右子节点[30]                                      │
│   -> [30]未满，插入后变为[25,30]                                      │
│   -> 最终树: 根[10,20], 左子[5,6], 中子[12], 右子[25,30]              │
└─────────────────────────────────────────────────────────────────────┘
    """)

    # 最终树形图标注
    print("\n" + "=" * 70)
    print("最终树形图（带标注）")
    print("=" * 70)
    print("""
                     ┌─────────────────┐
                     │    根节点       │
                     │   keys: [10,20] │
                     │ children: 3个   │
                     └────────┬────────┘
                              │
              ┌───────────────┼───────────────┐
              │               │               │
              ▼               ▼               ▼
        ┌──────────┐    ┌──────────┐    ┌──────────┐
        │ 左子节点 │    │ 中子节点 │    │ 右子节点 │
        │[5, 6]    │    │ [12]     │    │ [25,30]  │
        │ 叶子节点 │    │ 叶子节点 │    │ 叶子节点 │
        │ 2个键    │    │ 1个键    │    │ 2个键    │
        │ 0个子节点│    │ 0个子节点│    │ 0个子节点│
        └──────────┘    └──────────┘    └──────────┘

    【统计信息】
    - 总节点数: 4
    - 总键数: 7
    - 树的高度: 2
    - 根节点: 2个键，3个子节点 ✓
    - 非根内部节点: 无（所有非根节点都是叶子）✓
    - 所有叶子节点在同一层（深度=1）✓
    - 每个叶子节点键数在[1,2]范围内 ✓
    """)

    print("\n" + "=" * 70)
    print("作业完成！")
    print("=" * 70)


if __name__ == "__main__":
    main()

3阶 B-Tree 构建
插入序列: [10, 20, 5, 6, 12, 30, 25]

插入: 10
  插入到叶子节点: [10]
┌─ 节点 (keys=[10], leaf=True)

插入: 20
  插入到叶子节点: [10, 20]
┌─ 节点 (keys=[10, 20], leaf=True)

插入: 5
根节点已满，需要分裂
    分裂节点 [10, 20]，中间键 = 20
    分裂后左子节点: [10]
    分裂后右子节点: []
    父节点变为: [20]
  进入第 0 个子节点
  插入到叶子节点: [5, 10]
┌─ 节点 (keys=[20], leaf=False)
│  键 20 的左侧子树:
    ┌─ 节点 (keys=[5, 10], leaf=True)
├─ 键 20 的右侧子树:
    ┌─ 节点 (keys=[], leaf=True)
    ┌─ 节点 (keys=[], leaf=True)

插入: 6
  进入第 0 个子节点
  子节点 [5, 10] 已满，需要分裂
    分裂节点 [5, 10]，中间键 = 10
    分裂后左子节点: [5]
    分裂后右子节点: []
    父节点变为: [10, 20]
  插入到叶子节点: [5, 6]
┌─ 节点 (keys=[10, 20], leaf=False)
│  键 10 的左侧子树:
    ┌─ 节点 (keys=[5, 6], leaf=True)
├─ 键 10 的右侧子树:
│  键 20 的左侧子树:
    ┌─ 节点 (keys=[], leaf=True)
├─ 键 20 的右侧子树:
    ┌─ 节点 (keys=[], leaf=True)
    ┌─ 节点 (keys=[], leaf=True)

插入: 12
根节点已满，需要分裂
    分裂节点 [10, 20]，中间键 = 20
    分裂后左子节点: [10]
    分裂后右子节点: []
    父节点变为: [20]
  进入第 0 个子节点
  进入第 1 个子节点
  插入到叶子节点: [12]
┌─ 节点 (keys=[20], leaf=False)
│  键 20 的左侧子树:
    ┌─ 节点 (